# Data Exploration — chạy thử pipeline `qshield_data` từng bước

Notebook này **không** chứa logic tài chính hay logic xử lý dữ liệu riêng — nó chỉ gọi lại đúng các
hàm trong `packages/data/src/qshield_data/` theo thứ tự y hệt `qshield-data build`
(xem `packages/data/src/qshield_data/cli.py`), để chạy tay từng bước và xem output ở giữa chừng.
Đúng quy tắc CLAUDE.md: *"notebook không được chứa core logic — chỉ gọi lại hàm trong `packages/`"*.

**8 bước** (khớp `cli.py`):

1-2. Fetch — tải giá thô từ Yahoo/DNSE/vnstock
3. Clean — chuẩn hoá, dedup, loại phantom day, back-adjust corporate action đã đăng ký, loại pre-listing
4. Features — tính returns + market features
5. Eligibility — gắn cờ đủ điều kiện giao dịch mỗi ngày
6. Split — gán train/validation/test (2 đồng hồ: market & asset)
7. Quality — 6 check bắt buộc (DQ-001..006) phải PASS + 1 check cảnh báo (DQ-007, biên độ giá)
8. Manifest — data dictionary + hash/version

Notebook này **dừng ở bước 8 (Manifest)** — chưa nối sang `packages/ai` (regime/scenarios), đó là
bước tiếp theo sau khi phần data này chạy sạch.

Muốn chạy lại từ đầu trên máy sạch (không cần notebook): `uv run qshield-data build --config configs/base.yaml`
(cần `uv sync --all-packages` trước, xem `docs/runbook/setup.md`).

⚠️ Cell fetch gọi API thật (Yahoo/DNSE/vnstock) — cần internet, có thể mất vài phút và có thể lỗi tạm
thời do rate-limit; cứ chạy lại cell đó nếu vậy.

⚠️ **Bước 3 (Clean) áp dụng back-adjustment cho corporate action đã đăng ký thủ công** trong
`configs/data.yaml` (`corporate_actions`) — hiện có 1 entry: VCB chia cổ tức cổ phiếu ~49,5% ngày
2025-03-03 mà Yahoo không tự điều chỉnh, từng gây scenario validation gate (`kurtosis`, regime
`volatile`) FAIL. Điều tra đầy đủ + kiểm định sau khi sửa: `docs/perf/2026-08-05-kurtosis-fail-vcb.md`.

In [1]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import yfinance as yf
from qshield_contracts.config import Config


def _find_project_root(marker: str = "CLAUDE.md") -> Path:
    # Dò lên theo marker thay vì đếm cứng số cấp (Path.cwd().parents[1]) — cách đếm cứng chỉ đúng ở
    # lần chạy đầu tiên; nếu Cell này chạy lại lần 2 trong cùng kernel (không restart), cwd đã đổi
    # thành PROJECT_ROOT từ os.chdir() bên dưới, nên parents[1] sẽ nhảy lên tận /Users/mac — sai.
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Không tìm thấy {marker} từ {p} trở lên — notebook phải nằm trong repo QSHIELD."
    )


PROJECT_ROOT = _find_project_root()

# cwd mặc định của kernel Jupyter là thư mục chứa .ipynb (notebooks/exploration/), không phải repo
# root. _Paths (qshield_data.cli) build "data/", "reports/" là đường dẫn TƯƠNG ĐỐI, giả định process
# chạy từ repo root (đúng cho `uv run qshield-data ...`). Thiếu dòng chdir này notebook sẽ tạo nhầm
# notebooks/exploration/data/, notebooks/exploration/reports/ thay vì data/, reports/ ở gốc repo.
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"

cfg = Config.load(CONFIG_PATH)
date_range = cfg["date_range"]
cfg

{'expected_ticker_count': 8,
 'tickers': [{'ticker': 'ACB',
   'yahoo_symbol': 'ACB.VN',
   'company_name': 'Ngân hàng Á Châu',
   'first_trading_date': '2006-11-21',
   'exchange_current': 'HOSE',
   'exchange_history': 'HNX→HOSE 2020-12',
   'exchange_periods': [{'exchange': 'HNX', 'until': '2020-11-30'},
    {'exchange': 'HOSE', 'from': '2020-12-01'}],
   'data_source': 'dnse',
   'notes': ''},
  {'ticker': 'CTG',
   'yahoo_symbol': 'CTG.VN',
   'company_name': 'VietinBank',
   'first_trading_date': '2009-07-16',
   'exchange_current': 'HOSE',
   'exchange_history': '-',
   'exchange_periods': [{'exchange': 'HOSE'}],
   'data_source': 'yahoo',
   'notes': ''},
  {'ticker': 'VCB',
   'yahoo_symbol': 'VCB.VN',
   'company_name': 'Vietcombank',
   'first_trading_date': '2009-06-30',
   'exchange_current': 'HOSE',
   'exchange_history': '-',
   'exchange_periods': [{'exchange': 'HOSE'}],
   'data_source': 'yahoo',
   'notes': ''},
  {'ticker': 'HPG',
   'yahoo_symbol': 'HPG.VN',
   'com

> **Vì sao có `os.chdir(PROJECT_ROOT)` ở cell trên:** cwd mặc định của kernel Jupyter là thư mục
> chứa file `.ipynb` (`notebooks/exploration/`), không phải gốc repo. `_Paths` (trong `cli.py`) build
> `data/`, `reports/` là **đường dẫn tương đối** đọc từ `configs/base.yaml`, giả định process chạy từ
> repo root — đúng cho `uv run qshield-data ...` nhưng sai trong notebook nếu không `chdir`. Thiếu
> dòng này đã từng khiến notebook tạo nhầm `notebooks/exploration/data/` và
> `notebooks/exploration/reports/` thay vì `data/`, `reports/` ở gốc repo (đúng theo
> `docs/Structure.md` §1.1). Nếu restart kernel, luôn chạy lại cell đầu tiên trước khi nhảy vào cell khác.

In [2]:
from qshield_data.cli import _Paths

paths = _Paths(cfg)
paths.ensure()
paths.data_root, paths.raw_dir, paths.processed_dir, paths.metadata_dir

(PosixPath('data'),
 PosixPath('data/raw'),
 PosixPath('data/processed'),
 PosixPath('data/metadata'))

## Bước 1-2: Fetch

Đọc `configs/universe.yaml` (qua `cfg`) rồi tải giá 8 mã đã khóa phạm vi + VN-Index. `_Paths` khác
`qshield_contracts.paths.ArtifactPaths` — đây chỉ là glue path cho `data/`, `reports/` (không version
theo run_id, chỉ có một bộ "chính thức"); `ArtifactPaths` mới là nơi ghi `artifacts/runs/...`
(CLAUDE.md quy tắc 8).

Output: `data/raw/prices/*.csv`, `data/raw/vn_index/*.csv`,
`data/metadata/{universe_asof_*,source_register}.csv`, `data/metadata/raw_manifest.csv`.

In [3]:
from qshield_data.sources import registry

universe = registry.load_universe(cfg)
universe.head(10)

,ticker,yahoo_symbol,company_name,first_trading_date,exchange_current,exchange_history,exchange_periods,data_source,notes
0,ACB,ACB.VN,Ngân hàng Á Châu,2006-11-21,HOSE,HNX→HOSE 2020-12,"[{'exchange': 'HNX', 'until': '2020-11-30'}, {...",dnse,
1,CTG,CTG.VN,VietinBank,2009-07-16,HOSE,-,[{'exchange': 'HOSE'}],yahoo,
2,VCB,VCB.VN,Vietcombank,2009-06-30,HOSE,-,[{'exchange': 'HOSE'}],yahoo,
3,HPG,HPG.VN,Hoà Phát,2007-11-15,HOSE,-,[{'exchange': 'HOSE'}],yahoo,
4,VIC,VIC.VN,Vingroup,2007-09-19,HOSE,-,[{'exchange': 'HOSE'}],yahoo,
5,MWG,MWG.VN,Thế Giới Di Động,2014-07-14,HOSE,-,[{'exchange': 'HOSE'}],yahoo,
6,VNM,VNM.VN,Vinamilk,2006-01-19,HOSE,-,[{'exchange': 'HOSE'}],yahoo,
7,FPT,FPT.VN,FPT Corporation,2006-12-13,HOSE,-,[{'exchange': 'HOSE'}],yahoo,


In [4]:
from qshield_data.sources import fetch as fetch_mod

start = date_range["market_train_start"]
end = date_range["test_end"]

raw_manifest = fetch_mod.fetch_all_prices(
    universe, start=start, end=end, raw_dir=paths.raw_dir
)
raw_manifest["status"].value_counts()

status
OK    8
Name: count, dtype: int64

In [5]:
from qshield_data.cli import _raw_manifest_path

raw_manifest.to_csv(_raw_manifest_path(paths), index=False, encoding="utf-8-sig")
raw_manifest.head()

,ticker,symbol,source,preferred,fallback_used,file,rows,start,end,sha256,status
0,ACB,dnse:ACB,DNSE_PRICES,dnse,False,data/raw/prices/20260806_dnse_acb.csv,2637,2016-01-04,2026-07-30,786c40d1da996a602309feb00996d2bbaa4ab1fd41b756...,OK
1,CTG,CTG.VN,YF_PRICES,yahoo,False,data/raw/prices/20260806_yfinance_ctg.csv,2737,2016-01-04,2026-07-30,92a464aa287c04ca08a9f59866f2d1721b8770c262e2fb...,OK
2,VCB,VCB.VN,YF_PRICES,yahoo,False,data/raw/prices/20260806_yfinance_vcb.csv,2728,2016-01-04,2026-07-30,2011a19ccb71a59c62accd9d19c1eb494df310d2554b73...,OK
3,HPG,HPG.VN,YF_PRICES,yahoo,False,data/raw/prices/20260806_yfinance_hpg.csv,2737,2016-01-04,2026-07-30,f0e32674a75b885f74261ca3b2de410c7184495ed28fe2...,OK
4,VIC,VIC.VN,YF_PRICES,yahoo,False,data/raw/prices/20260806_yfinance_vic.csv,2737,2016-01-04,2026-07-30,3ad23c5f8b7b4982f42c969af894884ad7f7f6daa78899...,OK


In [6]:
index_df, symbol_used, source_used = fetch_mod.fetch_vn_index(
    start=start, end=end, raw_dir=paths.raw_dir
)
symbol_used, source_used

2026-08-06 20:04:22 - vnstock.common.data - INFO - Not a stock. Company and finance data unavailable.



  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')



('VNINDEX', 'vnstock_VCI')

In [7]:
import json

from qshield_data.cli import _vn_index_meta_path

_vn_index_meta_path(paths).write_text(
    json.dumps(
        {"symbol_used": symbol_used, "source_used": source_used}, ensure_ascii=False
    ),
    encoding="utf-8",
)
index_df.head() if index_df is not None else "⚠ VN-Index unavailable — features sẽ dùng custom composite"

,Open,High,Low,Close,Volume
date,,,,,
2015-07-16,627.32,632.21,622.57,626.90,113628750
2015-07-17,628.45,631.67,626.21,628.63,97179130
2015-07-20,624.84,624.84,616.11,620.54,105000850
2015-07-21,619.90,624.47,613.83,616.61,95668300
2015-07-22,614.78,629.94,613.58,629.85,109080570


In [8]:
from qshield_data.cli import _vnstock_version

data_cfg = cfg.get("data", {})
sources_df = registry.build_source_register(
    access_date=datetime.now().astimezone().strftime("%Y-%m-%d"),
    yfinance_version=yf.__version__,
    vnstock_version=_vnstock_version(),
    test_end=end,
)
universe_path, sources_path = registry.save_universe_and_sources(
    universe,
    sources_df,
    paths.metadata_dir,
    universe_as_of=data_cfg.get("universe_as_of", ""),
)
universe_path, sources_path

(PosixPath('data/metadata/universe_asof_20260803.csv'),
 PosixPath('data/metadata/source_register.csv'))

## Bước 3: Clean

`normalize` → `apply_registered_adjustments` → `dedup_prices` → `remove_yahoo_phantom_days` →
`flag_price_quality` → `remove_pre_listing`, đúng thứ tự trong `cli.py::clean()`. Không bước nào tự
ý xóa outlier (CLAUDE.md quy tắc 5) — `flag_price_quality` chỉ **gắn cờ** `quality_flag`, không xóa
dòng dữ liệu xấu; `apply_registered_adjustments` cũng không xóa gì, chỉ back-adjust `adjusted_close`
cho đúng sự kiện corporate action đã xác nhận thủ công trong `configs/data.yaml`.

Output: `data/processed/prices_adjusted.parquet`.

In [9]:
from qshield_data.clean import corporate_actions, normalize, validate_prices

raw_manifest = pd.read_csv(_raw_manifest_path(paths))
data_version = data_cfg.get("data_version", "v0.0.0")

prices = normalize.load_and_normalize(raw_manifest, data_version=data_version)
print(len(prices), prices["ticker"].nunique())
prices.head()

21769 8


,date,ticker,open,high,low,close,adjusted_close,volume,source_id,data_version
0,2016-01-04,ACB,2710.0,2710.0,2680.0,2690.0,2690.0,36439,DNSE_PRICES,v1.0.0
1,2016-01-05,ACB,2680.0,2680.0,2640.0,2650.0,2650.0,47676,DNSE_PRICES,v1.0.0
2,2016-01-06,ACB,2650.0,2660.0,2640.0,2660.0,2660.0,40589,DNSE_PRICES,v1.0.0
3,2016-01-07,ACB,2640.0,2650.0,2590.0,2610.0,2610.0,94027,DNSE_PRICES,v1.0.0
4,2016-01-08,ACB,2590.0,2610.0,2590.0,2610.0,2610.0,84809,DNSE_PRICES,v1.0.0


In [10]:
registered_actions = cfg.get("corporate_actions") or []
if registered_actions:
    prices = corporate_actions.apply_registered_adjustments(prices, registered_actions)
    print(
        f"Corporate action back-adjustment: {len(registered_actions)} entry đã đăng ký"
    )

is_vcb = prices["ticker"] == "VCB"
in_window = prices["date"].between("2025-02-25", "2025-03-05")
prices.loc[is_vcb & in_window, ["date", "close", "adjusted_close"]]

corporate_actions: back-adjust 2360 phiên của VCB trước 2025-03-03 theo hệ số 1.4950 (xem evidence trong configs/data.yaml).


Corporate action back-adjustment: 1 entry đã đăng ký


,date,close,adjusted_close
7730,2025-02-25,92600.000000,60985.378344
7731,2025-02-26,92300.000000,60787.803094
7732,2025-02-27,94000.000000,61907.399666
7733,2025-02-28,93300.000000,61446.389005
7734,2025-03-03,62408.027344,61446.390625
7735,2025-03-04,62207.359375,61248.812500
7736,2025-03-05,62207.359375,61248.812500


In [11]:
prices, n_dup = validate_prices.dedup_prices(prices)
n_dup

1

In [12]:
from qshield_data.cli import _load_latest_vn_index

vn_index = _load_latest_vn_index(paths)
prices, n_phantom = validate_prices.remove_yahoo_phantom_days(prices, vn_index)
n_phantom

680

In [13]:
prices = validate_prices.flag_price_quality(prices)
prices["quality_flag"].value_counts()

quality_flag
OK             20619
ZERO_VOLUME      469
Name: count, dtype: int64

In [14]:
prices = corporate_actions.remove_pre_listing(prices, universe)
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

out_path = paths.processed_dir / "prices_adjusted.parquet"
prices.to_parquet(out_path, index=False)
print(
    f"{len(prices):,} rows | {prices['ticker'].nunique()} tickers | "
    f"{prices['date'].min().date()} → {prices['date'].max().date()}"
)
prices.head()

21,088 rows | 8 tickers | 2016-01-04 → 2026-07-30


,date,ticker,open,high,low,close,adjusted_close,volume,source_id,data_version,quality_flag,turnover_value
0,2016-01-04,ACB,2710.0,2710.0,2680.0,2690.0,2690.0,36439,DNSE_PRICES,v1.0.0,OK,98020910.0
1,2016-01-05,ACB,2680.0,2680.0,2640.0,2650.0,2650.0,47676,DNSE_PRICES,v1.0.0,OK,126341400.0
2,2016-01-06,ACB,2650.0,2660.0,2640.0,2660.0,2660.0,40589,DNSE_PRICES,v1.0.0,OK,107966740.0
3,2016-01-07,ACB,2640.0,2650.0,2590.0,2610.0,2610.0,94027,DNSE_PRICES,v1.0.0,OK,245410470.0
4,2016-01-08,ACB,2590.0,2610.0,2590.0,2610.0,2610.0,84809,DNSE_PRICES,v1.0.0,OK,221351490.0


## Bước 4: Features

`compute_asset_returns` tính simple/log return **không forward-fill** (CLAUDE.md quy tắc 3 — ngày
đầu mỗi ticker là NaN, đúng ý, không che dữ liệu thiếu). `build_market_features` chỉ dùng dữ liệu
đến ngày `t` (point-in-time, quy tắc 4) — có `_assert_point_in_time` bên trong tự kiểm tra rolling
không nhìn thấy tương lai.

Output: `data/processed/returns.parquet`, `data/processed/market_features.parquet`.

In [15]:
from qshield_data import returns as returns_mod

prices = pd.read_parquet(paths.processed_dir / "prices_adjusted.parquet")
returns = returns_mod.compute_asset_returns(prices)
returns.to_parquet(paths.processed_dir / "returns.parquet", index=False)
returns.head()

,date,ticker,open,high,low,close,adjusted_close,volume,source_id,data_version,quality_flag,turnover_value,prev_adj,simple_return,log_return
0,2016-01-04,ACB,2710.0,2710.0,2680.0,2690.0,2690.0,36439,DNSE_PRICES,v1.0.0,OK,98020910.0,NaN,NaN,NaN
1,2016-01-05,ACB,2680.0,2680.0,2640.0,2650.0,2650.0,47676,DNSE_PRICES,v1.0.0,OK,126341400.0,2690.0,-0.014870,-0.014982
2,2016-01-06,ACB,2650.0,2660.0,2640.0,2660.0,2660.0,40589,DNSE_PRICES,v1.0.0,OK,107966740.0,2650.0,0.003774,0.003766
3,2016-01-07,ACB,2640.0,2650.0,2590.0,2610.0,2610.0,94027,DNSE_PRICES,v1.0.0,OK,245410470.0,2660.0,-0.018797,-0.018976
4,2016-01-08,ACB,2590.0,2610.0,2590.0,2610.0,2610.0,84809,DNSE_PRICES,v1.0.0,OK,221351490.0,2610.0,0.000000,0.000000


In [16]:
from qshield_data import features as features_mod
from qshield_data.cli import _load_vn_index_source_tag

index_df = _load_latest_vn_index(paths)
index_source_tag = _load_vn_index_source_tag(paths)
market = features_mod.build_market_features(index_df, returns, index_source_tag)
market.to_parquet(paths.processed_dir / "market_features.parquet", index=False)
market.head()

,date,open,high,low,close,volume,source,market_log_return,market_simple_return,realized_vol_20d,rolling_max_252,drawdown,liquidity_20d
0,2015-07-16,627.32,632.21,622.57,626.90,113628750,vnstock_VCI_VNINDEX,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-07-17,628.45,631.67,626.21,628.63,97179130,vnstock_VCI_VNINDEX,0.002756,0.002760,NaN,NaN,NaN,NaN
2,2015-07-20,624.84,624.84,616.11,620.54,105000850,vnstock_VCI_VNINDEX,-0.012953,-0.012869,NaN,NaN,NaN,NaN
3,2015-07-21,619.90,624.47,613.83,616.61,95668300,vnstock_VCI_VNINDEX,-0.006353,-0.006333,NaN,NaN,NaN,NaN
4,2015-07-22,614.78,629.94,613.58,629.85,109080570,vnstock_VCI_VNINDEX,0.021245,0.021472,NaN,NaN,NaN,NaN


## Bước 5: Eligibility

Gắn cờ mỗi ngày mỗi mã có đủ điều kiện giao dịch không (đủ lịch sử, đủ coverage, đủ thanh khoản 20
ngày) — tính point-in-time cumulative dựa trên `first_data_date` thực tế, không nhìn tương lai.

Output: `data/processed/eligibility_daily.parquet`.

In [17]:
from qshield_data import eligibility as eligibility_mod

elig_cfg = cfg.get("eligibility", {})
elig_df = eligibility_mod.build_eligibility(
    prices,
    universe,
    min_history_sessions=cfg["min_history_sessions"],
    min_coverage_pct=elig_cfg["min_coverage_pct"],
    min_turnover_20d_vnd=elig_cfg["min_turnover_20d_vnd"],
)
elig_df.to_parquet(paths.processed_dir / "eligibility_daily.parquet", index=False)
elig_df["reason_code"].value_counts()

reason_code
OK                      19077
INSUFFICIENT_HISTORY     2008
SUSPENDED_OR_NO_DATA       40
LOW_LIQUIDITY               3
Name: count, dtype: int64

## Bước 6: Split

"2 đồng hồ": `level="asset"` cho `returns` (train bắt đầu muộn hơn — coverage tốt hơn giữa các mã
trong universe), `level="market"` cho `market_features` (dùng cho HMM regime bên `packages/ai`).
`apply_splits` raise `ValueError` nếu khoảng train/validation/test trong config chồng lấn nhau
(PR-DAT-013) — kiểm tra trên **khoảng ngày của config**, không phải trên nhãn split đã gán (bug cũ:
so nhãn kết quả luôn rỗng vì mỗi ngày chỉ có đúng 1 nhãn, không bao giờ raise được — đã sửa trong
`split.py`).

In [18]:
from qshield_data import split as split_mod

returns_path = paths.processed_dir / "returns.parquet"
market_path = paths.processed_dir / "market_features.parquet"

returns = pd.read_parquet(returns_path)
returns["date"] = pd.to_datetime(returns["date"])
returns = split_mod.apply_splits(returns, level="asset", splits_config=date_range)
returns.to_parquet(returns_path, index=False)
returns["split"].value_counts()

split
train           9007
test            5120
out_of_scope    4969
validation      1992
Name: count, dtype: int64

In [19]:
market = pd.read_parquet(market_path)
market["date"] = pd.to_datetime(market["date"])
market = split_mod.apply_splits(market, level="market", splits_config=date_range)
market.to_parquet(market_path, index=False)
market["split"].value_counts()

split
train           1752
test             641
validation       249
out_of_scope     120
Name: count, dtype: int64

## Bước 7: Quality Gate

6 check bắt buộc (`DQ-001`…`DQ-006`): duplicate, giá âm/0, volume âm, pre-listing, đúng 8 mã trong
universe, split không chồng lấn. Toàn bộ phải **PASS** trước khi bàn giao cho Tú/Phúc (BR-020 trong
`cli.py`) — nếu FAIL, đọc `report_df` ở cell trước để biết đúng check nào lỗi rồi quay lại bước
tương ứng, đừng đi tiếp.

Cộng thêm `DQ-007` (`type=WARN`, không chặn gate): phát hiện return ngày vượt biên độ dao động của
sàn (`quality/price_limits.py`, xem `docs/perf/2026-08-05-kurtosis-fail-vcb.md`) — cần tính
`price_limit_violations` TRƯỚC khi gọi `run_all_checks` (tham số bắt buộc, không có mặc định),
đúng thứ tự `cli.py::quality()` thật đang làm.

Output: `reports/data_quality_report.csv`, `reports/price_limit_violations.csv`
(`docs/Structure.md` ghi `.html`, nhưng `plan.md` chốt CSV cho MVP — xem docstring `quality/report.py`).

In [20]:
from qshield_data.quality import checks as checks_mod
from qshield_data.quality import report as report_mod
from qshield_data.quality.price_limits import find_price_limit_violations

prices = pd.read_parquet(paths.processed_dir / "prices_adjusted.parquet")
returns = pd.read_parquet(returns_path)
expected_count = cfg.get("expected_ticker_count", len(universe))

price_limits_cfg = cfg["price_limits"]
violations = find_price_limit_violations(
    returns,
    universe,
    bands_by_exchange=price_limits_cfg["bands_by_exchange"],
    tolerance_pct=price_limits_cfg["tolerance_pct"],
)

report_df, all_pass = checks_mod.run_all_checks(
    prices, universe, returns, expected_count, violations
)
print("✅ DATA QUALITY GATE: PASS" if all_pass else "❌ DATA QUALITY GATE: FAIL")
report_df

✅ DATA QUALITY GATE: PASS


,check_id,check_name,type,status,count,trace
0,DQ-001,"No duplicate (date, ticker)",MUST_PASS,PASS,0,"AC-DAT-004, PR-DAT-006"
1,DQ-002,adjusted_close > 0 và không NaN,MUST_PASS,PASS,0,"AC-DAT-005, PR-DAT-007"
2,DQ-003,Không có volume âm,MUST_PASS,PASS,0,AC-DAT-005
3,DQ-004,Không có giá trước first_trading_date,MUST_PASS,PASS,0,"AC-DAT-011, PR-DAT-017"
4,DQ-005,Universe = 8 tickers,MUST_PASS,PASS,8,"AC-DAT-001, PR-DAT-001"
5,DQ-006,Train/Val/Test không giao nhau,MUST_PASS,PASS,0,"AC-DAT-009, PR-DAT-013"
6,DQ-007,Return ngày trong biên độ sàn,WARN,WARN,23,docs/perf/2026-08-05-kurtosis-fail-vcb.md


In [21]:
dq_out_path = paths.reports_root / "data_quality_report.csv"
report_mod.write_quality_report(report_df, dq_out_path)
dq_out_path

PosixPath('reports/data_quality_report.csv')

In [22]:
violations_path = paths.reports_root / "price_limit_violations.csv"
report_mod.write_violations(violations, violations_path)
print(f"DQ-007: {len(violations)} phiên vượt biên độ sàn → {violations_path}")
violations.head(10)

DQ-007: 23 phiên vượt biên độ sàn → reports/price_limit_violations.csv


,date,ticker,exchange,simple_return,band,tolerance,excess
0,2025-10-13,VIC,HOSE,0.144290,0.07,0.005,0.074290
1,2021-07-09,MWG,HOSE,0.125638,0.07,0.005,0.055638
2,2020-05-19,HPG,HOSE,0.121086,0.07,0.005,0.051086
3,2025-07-14,VIC,HOSE,0.112205,0.07,0.005,0.042205
4,2018-01-25,VCB,HOSE,0.109836,0.07,0.005,0.039836
5,2025-10-20,VNM,HOSE,-0.101307,0.07,0.005,0.031307
6,2021-07-09,HPG,HOSE,-0.095602,0.07,0.005,0.025602
7,2019-07-17,VCB,HOSE,0.094828,0.07,0.005,0.024828
8,2025-09-29,VIC,HOSE,0.093671,0.07,0.005,0.023671
9,2021-02-17,FPT,HOSE,0.092567,0.07,0.005,0.022567


## Bước 8: Manifest

`build_data_dictionary` sinh Excel mô tả cột của từng artifact; `build_manifest`/`write_manifest`
ghi hash SHA-256 + version + row count + cấu hình split/eligibility đã dùng để chạy — để Tú/Phúc
truy lại chính xác dữ liệu này sinh ra từ config/nguồn nào (CLAUDE.md quy tắc 13: chỉ metadata cấp
run mới do `RunContext` ghi; đây là metadata cấp *bộ dữ liệu*, do `manifest.py` ghi, khác phạm vi).

Output: `data/metadata/data_dictionary.xlsx`, `data/metadata/data_manifest.json`.

In [23]:
dict_out = paths.metadata_dir / "data_dictionary.xlsx"
report_mod.build_data_dictionary(dict_out)
dict_out

PosixPath('data/metadata/data_dictionary.xlsx')

In [24]:
from qshield_data.cli import _run_id
from qshield_data.manifest import build_manifest, write_manifest

universe_files = sorted(paths.metadata_dir.glob("universe_asof_*.csv"))
universe_register_path = (
    universe_files[-1]
    if universe_files
    else paths.metadata_dir / "universe_asof_MISSING.csv"
)

files = {
    "universe_register": universe_register_path,
    "source_register": paths.metadata_dir / "source_register.csv",
    "prices_adjusted": paths.processed_dir / "prices_adjusted.parquet",
    "returns": returns_path,
    "market_features": market_path,
    "eligibility_daily": paths.processed_dir / "eligibility_daily.parquet",
    "data_dictionary": dict_out,
    "data_quality_report": dq_out_path,
}

row_counts = {
    name: (len(pd.read_parquet(files[name])) if files[name].exists() else 0)
    for name in ("prices_adjusted", "returns", "market_features", "eligibility_daily")
}
row_counts["universe"] = len(universe)

dq_df = pd.read_csv(dq_out_path)
quality_gate_pass = bool(
    (dq_df[dq_df["type"] == "MUST_PASS"]["status"] == "PASS").all()
)

manifest_dict = build_manifest(
    run_id=_run_id(),
    data_version=data_cfg.get("data_version", "v0.0.0"),
    universe_version=data_cfg.get("universe_version", "v0.0"),
    universe_as_of=data_cfg.get("universe_as_of", ""),
    files=files,
    row_counts=row_counts,
    splits_config={
        "market": {
            "train": [date_range["market_train_start"], date_range["market_train_end"]],
            "validation": [
                date_range["validation_start"],
                date_range["validation_end"],
            ],
            "test": [date_range["test_start"], date_range["test_end"]],
        },
        "asset": {
            "train": [date_range["asset_train_start"], date_range["asset_train_end"]],
            "validation": [
                date_range["validation_start"],
                date_range["validation_end"],
            ],
            "test": [date_range["test_start"], date_range["test_end"]],
        },
    },
    eligibility_config={
        "min_history_sessions": cfg.get("min_history_sessions"),
        **cfg.get("eligibility", {}),
    },
    quality_gate_pass=quality_gate_pass,
    index_symbol_used=symbol_used,
    index_source_used=source_used,
    data_root=paths.data_root,
)
manifest_out = paths.metadata_dir / "data_manifest.json"
write_manifest(manifest_dict, manifest_out)
manifest_out

PosixPath('data/metadata/data_manifest.json')

## Xong — checklist đầu ra

Sau khi chạy hết notebook, kiểm tra các file này tồn tại đúng chỗ (**gốc repo**, không phải trong
`notebooks/exploration/`):

- `data/processed/{prices_adjusted,returns,market_features,eligibility_daily}.parquet`
- `data/metadata/{universe_asof_*,source_register,data_dictionary.xlsx,data_manifest.json,raw_manifest.csv}`
- `reports/data_quality_report.csv`
- `reports/price_limit_violations.csv` — kỳ vọng **23 dòng** (không còn VCB 2025-03-03) nếu
  `corporate_actions` trong `configs/data.yaml` đã áp dụng đúng, xem
  `docs/perf/2026-08-05-kurtosis-fail-vcb.md`.

Nếu bất kỳ file nào rơi vào `notebooks/exploration/data/` hoặc `notebooks/exploration/reports/` —
nghĩa là Cell 1 (`os.chdir`) chưa chạy trong phiên kernel hiện tại. Restart kernel → chạy lại từ
Cell 1, đừng nhảy cóc.

**Bước tiếp theo (chưa có trong notebook này):** nối sang `packages/ai` (`regime` rồi `scenarios`)
trên bộ dữ liệu vừa build — làm sau khi phần data này đã chạy sạch, không gộp chung vào đây.